# `RunnableGenerator: Runnable[Input, Output]`

`RunnableGenerator` wraps a synchronous or asynchronous generator function as a `Runnable`.

It is used when a component must process an incoming stream and emit output chunks without waiting for the entire previous step to finish.

## Type Parameters

```python
Input # Type of each input chunk received by the generator
Output # Type of each output chunk yielded by the generator
```

## Field

```python
name: str # Runnable name used for tracing and representation
```

## Constructor

```python
RunnableGenerator(
    transform: Callable[[Iterator[Input]], Iterator[Output]]
    | Callable[[AsyncIterator[Input]], AsyncIterator[Output]], # Synchronous or asynchronous generator function
    atransform: Callable[[AsyncIterator[Input]], AsyncIterator[Output]] | None = None, # Optional asynchronous implementation
    *,
    name: str | None = None, # Optional custom Runnable name
) -> None # Initialize the RunnableGenerator
```

The constructor raises `TypeError` when `transform` is not a synchronous or asynchronous generator function.

## Generator Requirements

A synchronous generator receives an `Iterator[Input]` and yields `Output` chunks.

```python
Callable[[Iterator[Input]], Iterator[Output]] # Required synchronous generator form
```

An asynchronous generator receives an `AsyncIterator[Input]` and asynchronously yields `Output` chunks.

```python
Callable[[AsyncIterator[Input]], AsyncIterator[Output]] # Required asynchronous generator form
```

When both implementations are supplied, synchronous operations use `transform` and asynchronous operations use `atransform`.

## Overridden Properties and Methods

### `InputType`

Infers the input type from the item type inside the generator function's first iterator parameter.

Returns `Any` when the type cannot be inferred.

### `get_input_schema`

Returns a schema based on the inferred input type and the module containing the generator function.

### `OutputType`

Infers the output type from the item type inside the generator function's return annotation.

Returns `Any` when the type cannot be inferred.

### `get_output_schema`

Returns a schema based on the inferred output type and the module containing the generator function.

### `__eq__`

Returns `True` when two `RunnableGenerator` objects wrap the same synchronous generator or the same asynchronous generator.

### `__repr__`

Returns a representation containing the Runnable name.

### `transform`

Processes a synchronous input iterator and yields output chunks as they are generated.

Raises `NotImplementedError` when no synchronous generator implementation is available.

### `stream`

Wraps one input value as an iterator and synchronously yields output chunks through `transform()`.

### `invoke`

Consumes the synchronous output stream and combines all yielded chunks into one final output.

### `atransform`

Processes an asynchronous input iterator and asynchronously yields output chunks as they are generated.

Raises `NotImplementedError` when no asynchronous generator implementation is available.

### `astream`

Wraps one input value as an asynchronous iterator and yields output chunks through `atransform()`.

### `ainvoke`

Consumes the asynchronous output stream and combines all yielded chunks into one final output.

## Inherited Execution Methods

### `batch`

Runs `invoke()` for multiple inputs and returns the combined output for each input.

### `abatch`

Runs `ainvoke()` for multiple inputs and returns the combined output for each input.

### `batch_as_completed`

Yields indexed synchronous results as individual inputs finish.

### `abatch_as_completed`

Asynchronously yields indexed results as individual inputs finish.

## Behaviour

- `stream()` and `astream()` expose individual output chunks.
- `invoke()` and `ainvoke()` combine output chunks using the `+` operator.
- Output chunks must support addition when more than one chunk is produced.
- A sync-only generator supports synchronous execution methods.
- An async-only generator supports asynchronous execution methods.
- The class is unhashable because equality is based on the wrapped generator function.

In [ ]:
from collections.abc import Iterator # Import Iterator type
from langchain_core.runnables import RunnableGenerator # Import RunnableGenerator


def uppercase_stream(inputs: Iterator[str]) -> Iterator[str]: # Process incoming text chunks
    for text in inputs: # Read each input chunk
        yield text.upper() # Yield each transformed chunk
    return # End the generator


runnable = RunnableGenerator(uppercase_stream) # Wrap the generator as a Runnable

for chunk in runnable.stream("hello"): # Stream transformed output chunks
    print(chunk) # Display each output chunk